# Treino YOLO11n no Colab

Fluxo: `python pipeline.py sync` (local) envia `dataset.zip` ao Drive -> este notebook treina com kernel **Colab**
(VS Code: seletor de kernel -> Colab -> New Colab Server -> GPU T4) -> `python pipeline.py fetch` (local) traz os resultados.

Rode as células em ordem. Se o runtime cair, **rode tudo de novo**: o treino retoma do `last.pt` salvo no Drive.

In [6]:
# ---- Parâmetros (mantenha DRIVE_DIR/RUN_NAME iguais a [colab] em configs/pipeline.toml) ----
DRIVE_DIR = "/content/drive/MyDrive/TCC"
RUN_NAME  = "yolo11n_v1"
MODEL     = "yolo11n.pt"   # pesos COCO (transfer learning)
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 16
PATIENCE  = 30             # early stopping
SEED      = 0
MIN_MAP50 = 0.80           # limiar de aprovação do TCC

In [2]:
# ---- GPU ----
import subprocess, torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print("SEM GPU: troque o runtime (kernel Colab -> New Colab Server -> GPU) antes de treinar.")

CUDA disponível: True
name, memory.total [MiB]
Tesla T4, 15360 MiB



In [3]:
# ---- Google Drive ----
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
%pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 12.7 MB/s eta 0:00:00


In [9]:
# ---- Dataset: copia do Drive para o disco local do Colab (muito mais rápido que ler do Drive) ----
import hashlib, pathlib, shutil, zipfile, yaml

DST = pathlib.Path("/content/dataset")
src_zip = pathlib.Path(DRIVE_DIR) / "dataset.zip"
assert src_zip.exists(), f"{src_zip} não existe. Rode 'python pipeline.py sync' e espere o Drive sincronizar."

local_zip = pathlib.Path("/content/dataset.zip")
shutil.copy(src_zip, local_zip)

sha_file = pathlib.Path(DRIVE_DIR) / "dataset.zip.sha256"
if sha_file.exists():
    h = hashlib.sha256(local_zip.read_bytes()).hexdigest()
    assert h == sha_file.read_text().strip(), "dataset.zip corrompido ou sincronização incompleta no Drive."
    print("sha256 OK")

shutil.rmtree(DST, ignore_errors=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(DST)

cfg = yaml.safe_load((DST / "dataset.yaml").read_text())
cfg["path"] = str(DST)                       # caminho absoluto no Colab
(DST / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
for s in ("train", "val", "test"):
    n = len(list((DST / "images" / s).glob("*.jpg"))) if (DST / "images" / s).exists() else 0
    print(f"{s}: {n} imagens")
print("classes:", cfg["names"])

train: 751 imagens
val: 216 imagens
test: 110 imagens
classes: {0: 'amortecedor_vibracao', 1: 'esfera_sinalizacao', 2: 'isolador_bastao', 3: 'isolador_disco'}


In [10]:
# ---- Treino (retoma sozinho se já existir last.pt no Drive) ----
from ultralytics import YOLO

runs_dir = pathlib.Path(DRIVE_DIR) / "runs"
run_dir  = runs_dir / RUN_NAME
last     = run_dir / "weights" / "last.pt"

if last.exists():
    print("Retomando de", last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DST / "data.yaml"),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
        project=str(runs_dir), name=RUN_NAME, exist_ok=True,
        workers=2, cache=True,   # Colab tem poucos núcleos; cache em RAM acelera
        # augmentation: padrão do Ultralytics (mosaic, flip horizontal, HSV, escala, translação)
    )

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.181 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, free

In [11]:
# ---- Avaliação final no conjunto de TESTE (use apenas uma vez, no fim) ----
import json
best = YOLO(str(run_dir / "weights" / "best.pt"))
m = best.val(data=str(DST / "data.yaml"), split="test", imgsz=IMGSZ,
             project=str(runs_dir), name=RUN_NAME + "_test", exist_ok=True)

res = {
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "per_class_mAP50": {m.names[int(i)]: float(v) for i, v in zip(m.box.ap_class_index, m.box.ap50)},
}
(run_dir / "test_metrics.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))
print(json.dumps(res, indent=2, ensure_ascii=False))
print("\nAPROVADO" if res["mAP50"] >= MIN_MAP50 else "\nREPROVADO", f"(mAP@0.5 = {res['mAP50']:.3f}, limiar {MIN_MAP50})")
print("Agora, no terminal local: python pipeline.py fetch")

Ultralytics 8.4.181 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1968.9±321.2 MB/s, size: 76.8 KB)
val: Scanning /content/dataset/labels/test... 110 images, 52 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 110/110 2.7Kit/s 0.0s
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.1it/s 2.2s0.2ss
                   all        110        108      0.983      0.969      0.991      0.736
  amortecedor_vibracao         30         30      0.966      0.936      0.985      0.684
    esfera_sinalizacao         27         27      0.984          1      0.995      0.919
       isolador_bastao         34         34      0.998          1      0.995      0.595
        isolador_disco         17         17      0.985      0.941      0.987  